# Resource usage versus asks: friction V0 runs

Scratch audit of the jobmon runs in the friction node (`friction_dir` in `paths.yaml`):
the full run in the `current` snapshot (frozen from `working/` as 20261105) and the probes in `scratch/probe_v0/`.

The functions live in `idd_travel_time.friction.audit` (`task_usage`, `attempts_by_state`, `usage_summary`, `usage_by_tier`); this notebook only calls them.

Measures per task:

- **asked**: memory and wall limit the task ran under, from sacct (`req_mem_gib`, `timelimit_minutes` in the slot's `audit*.csv`, written by `build-friction audit`);
- **used, memory**: the largest per-block process peak RSS of the task (REPORT lines of every attempt, `launch.report_peaks`), not sacct's MaxRSS, which is the cgroup peak and fills with file cache up to the ask;
- **used, runtime**: the task's wall time of its last attempt (sacct `elapsed_seconds`) and its block compute time (sum of `seconds_total` over the blocks it registered in its atomic sidecar, `launch.block_compute_minutes`);
- **group**: the task's memory tier (`mem_tier` in its task features, from the predicted block peak).

The criterion (Bobby, 2026-11-03): the mean used/asked per axis near 0.65, no tier systematically off, a small rerun rate.

In [ ]:
from pathlib import Path

import json
import pandas as pd

from idd_travel_time.friction import launch as ln
from idd_travel_time.paths import load_paths

pd.set_option("display.width", 200)
pd.set_option("display.precision", 2)
node = Path(load_paths()["friction_dir"])
# the frozen V0 snapshot (current) and the probes; a run still in working/ would be node / "working"
slots = {"current": node / "current", "probes": node / "scratch" / "probe_v0"}
{k: v.exists() for k, v in slots.items()}

In [ ]:
from idd_travel_time.friction.audit import task_usage

tasks = pd.concat([task_usage(slot) for slot in slots.values()], ignore_index=True)
tasks.groupby("workflow").size().rename("tasks_audited")

In [ ]:
from idd_travel_time.friction.audit import attempts_by_state

pd.concat([attempts_by_state(slot) for slot in slots.values()]).fillna(0).astype(int).sort_index()

## Overall, per workflow

`asked_hours` and `used_hours` are task-hours (sum of limits, sum of wall); `asked_gib_hours` and `peak_gib_hours` weight memory by time.

In [ ]:
from idd_travel_time.friction.audit import usage_summary

usage_summary(tasks)

## By memory tier, per workflow

In [ ]:
from idd_travel_time.friction.audit import usage_by_tier

usage_by_tier(tasks)

## Zones: over, correctly and under resourced

idd-tools' thresholds (`EfficiencyBins`: over < 50% used, well 50 to 80%, under > 80%) applied to the measured ratios: memory by process peak, time by wall. The combined `zone_*` columns: `zone_under` = under on either axis, `zone_well` = well on both, `zone_over` = over on at least one axis and under on none.

In [ ]:
from idd_travel_time.friction.audit import classify, zone_counts

classified = classify(tasks)
zone_counts(classified)

## Plots

All three show the full run (workflow 648580); pass `classified` instead of `full` for every workflow. Colours are the status and categorical colours of the dataviz reference palette; each zone and tier also carries its own marker.

In [ ]:
from idd_travel_time.friction.audit import plot_ratios, plot_usage, plot_usage_by_tier

full = classified[classified["workflow"] == 648580]
plot_usage(full);

In [ ]:
plot_usage_by_tier(full);

In [ ]:
plot_ratios(full);

## idd-tools' own summaries on the sacct-joined audit

`policy_metrics` and `sizing_floor_report` read the audit frame as `audit_run` wrote it (sacct zones: memory by MaxRSS, which fills to the ask here, so its memory zone reads "under"); `cost_by_feature` groups a cost frame by a feature, here fed the process peak and wall time per task.

In [ ]:
from idd_tools.jobmon import cost_by_feature
from idd_tools.jobmon.resource_audit import policy_metrics, sizing_floor_report

for wf in sorted(tasks["workflow"].unique()):
    slot = slots["current"] if wf == 648580 else slots["probes"]
    audit = ln.slot_audits(slot, workflows=[wf])
    print(f"workflow {wf}: sacct zones {policy_metrics(audit)['overall'].round(2).to_dict()}; "
          f"sizing floor {sizing_floor_report(audit)}")
cost = tasks.rename(columns={"peak_gib": "jobmon_mem_gib"}).assign(jobmon_runtime_s=tasks["wall_min"] * 60)
cost_by_feature(cost[cost["workflow"] == 648580], by="tier")

## Reading the tables

- Memory: judge `mem_used_over_asked` (process peak over ask). The sacct column `sacct_gib` sits at the ask on most tasks and says nothing about need.
- Runtime: `time_used_over_asked` is wall over limit for the last attempt; a task that timed out and resumed shows a short last attempt, so read it with the attempts table. `compute_min` is the uncensored block build time.
- A tier whose mean sits far from the others is a systematic miss of the model for that group.